In [1]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Markdown, display
import os
import winsound

# Monte Carlo Simulations of our System

## Code 

### Create the system

In [2]:
def wrap_angle(x):
    '''
    Wrap angle between -pi/6 and pi/6
    Inputs:
        x: angle
    Outputs:
        wrapped angle
    '''

    return (x + np.pi/6) % (np.pi/3) - np.pi/6

def theta_flatten(theta, indices):
    '''
    Flatten theta array
    Inputs:
        theta: 2D
        indices
    Outputs:
        theta_flat: 1D
    '''

    return np.array([theta[i, j] for (i, j) in indices])

class Lattice:
    '''
    Lattice class to hold lattice properties
    Attributes:
        L: size
        indices: list of (i,j) coordinates
        voisins: list of neighbors for each (i,j)
        theta: angles at each (i,j)
        rho: density
        distance: distance between points
        d_x, d_y: x and y distances
        xs, ys: x and y coordinates
        coords: array of (x,y) coordinates
        r: distance matrix
    '''

    def __init__(self, L, rho=1.0):
        self.L = L
        self.rho = rho
        self.indices = np.array([(i, j) for i in range(L) for j in range(L)])

        self.index_map = np.zeros((L, L), dtype=int)

        self.voisins = [[] for _ in range(L*L)]

        aire = self.L * self.L / self.rho
        self.distance = np.sqrt(aire) / self.L
        self.d_x = self.distance
        self.d_y = self.distance * np.sqrt(3) / 2
        xs, ys = [], []

        for idx, (i, j) in enumerate(self.indices):
            self.index_map[i, j] = idx

            x = j * self.d_x + (i % 2) * (self.d_x / 2)
            y = i * self.d_y
            xs.append(x)
            ys.append(y)

            if i % 2 == 0:
                candidats = [
                (i + 1, j),
                (i - 1, j),
                (i, j + 1),
                (i, j - 1),
                (i + 1, j - 1),
                (i - 1, j - 1)
            ]
            else:
                candidats = [
                (i + 1, j),
                (i - 1, j),
                (i, j + 1),
                (i, j - 1),
                (i + 1, j + 1),
                (i - 1, j + 1)
            ]
            v_valid = [(ni, nj) for (ni, nj) in candidats if 0 <= ni < L and 0 <= nj < L]
            self.voisins[idx] = v_valid

        self.xs = np.array(xs)
        self.ys = np.array(ys)
        self.coords = np.vstack((self.xs, self.ys)).T
        self.r = squareform(pdist(self.coords, metric='euclidean'))

        self.theta = np.random.uniform(-np.pi/6, np.pi/6, size=(L, L))

        self.dr = self.distance / 2
        r_max = self.distance * L
        self.n_bins = int(r_max / self.dr) + 1
        p = 2.0
        xs = np.linspace(0, 1, self.n_bins, endpoint=False)
        self.r_centers = r_max * (1 - (1 - xs)**p)
        self.bin_indices = np.digitize(self.r, self.r_centers) - 1

    def compute_psi6(self):
        '''
        Compute the local hexatic order parameter psi6
        Outputs:
            psi6: 2D array of psi6 values
        '''

        psi6 = np.exp(1j * 6 * self.theta)
        return psi6
    
    def compute_magnetization(self):
        '''
        Compute the magnetization of the lattice
        Outputs:
            m: magnetization
        '''

        psi6 = self.compute_psi6()
        psi6_flat = theta_flatten(psi6, self.indices)
        m = np.sum(psi6_flat) / len(psi6_flat)
        return m            

    def get_xy_positions(self, i, j):
        '''
        Get (x,y) of the (i,j) point 
        Inputs:
            i, j
        Outputs:
            x, y
        '''

        x = j * self.d_x + (i % 2) * (self.d_x / 2)
        y = i * self.d_y
        return x, y
    
    def hamiltonien(self, epsilon, gamma, A):
        '''
        Compute the Hamiltonian of the system
        Inputs:
            epsilon, gamma, A: parameters
        Outputs:
            Hamiltonian
        '''

        H_0 = (epsilon/self.rho) * np.sum(theta_flatten(self.theta, self.indices)**2)
        H_int = 0.0
        for idx, (i, j) in enumerate(self.indices):
            t_ij = self.theta[i, j]
            for (ni, nj) in self.voisins[idx]:
                dtheta = np.abs(wrap_angle(t_ij - self.theta[ni, nj]))
                H_int += dtheta * (A - np.log(max(dtheta, 1e-12)))
        H_int = (gamma / (3 * np.sqrt(self.rho))) * H_int
        H_int /= 2
        return H_0 + H_int

    def dhamiltonien(self, new_theta, idx, epsilon, gamma, A):
        '''
        Compute the change in Hamiltonian when one theta is changed
        Inputs:
            new_theta
            idx: index of the changed theta in indices
            epsilon, gamma, A: parameters
        Outputs:
            Change in Hamiltonian
        '''

        i, j = self.indices[idx]
        old_theta = self.theta[i, j]
        dH_0 = (epsilon/self.rho) * (new_theta**2 - old_theta**2)
        dH_int = 0.0
        for (ni, nj) in self.voisins[idx]:
            dtheta_old = np.abs(wrap_angle(old_theta - self.theta[ni, nj]))
            dtheta_new = np.abs(wrap_angle(new_theta - self.theta[ni, nj]))
            dH_int += dtheta_new * (A - np.log(max(dtheta_new, 1e-12))) - dtheta_old * (A - np.log(max(dtheta_old, 1e-12)))
        dH_int = (gamma / (3 * np.sqrt(self.rho))) * dH_int
        return dH_0 + dH_int
    
    def compute_orientationnal_correlation(self):
        '''
        Compute the correlation function of the lattice
        Outputs:
            G : correlation function
        '''
        
        psi = self.compute_psi6()
        psi_flat = theta_flatten(psi, self.indices)

        G = np.outer(psi_flat, np.conj(psi_flat))

        mask = np.triu(np.ones_like(self.r, dtype=bool), k=0)

        G_vals = G[mask]

        G_avg = np.zeros(self.n_bins, dtype=complex)
        counts = np.zeros(self.n_bins, dtype=int)

        np.add.at(G_avg, self.bin_indices[mask], G_vals)
        np.add.at(counts, self.bin_indices[mask], 1)

        G_avg /= np.maximum(counts, 1)
        G_avg = np.real(G_avg)

        return G_avg

### Monte Carlo Simulation

In [ ]:
class Metropolis:
    def __init__(self, delta_init=0.2, target_acceptance=0.5, tune_interval=100_000, growth_factor=1.05, shrink_factor=0.95, min_delta=1e-3, max_delta=np.pi/6):
        self.delta = delta_init
        self.target_acceptance = target_acceptance
        self.tune_interval = tune_interval
        self.growth_factor = growth_factor
        self.shrink_factor = shrink_factor
        self.min_delta = min_delta
        self.max_delta = max_delta
        self.attempts = 0
        self.acceptances = 0

    def metropolis_step(self, lattice, epsilon, gamma, A, beta, tune=True):
        '''
        Perform one Metropolis step with adaptive delta
        Inputs:
            lattice: Lattice object
            epsilon, gamma, A: parameters
            beta
        '''
        N = len(lattice.indices)
        idx = np.random.randint(N)
        i, j = lattice.indices[idx]
        new_theta = wrap_angle(lattice.theta[i, j] + np.random.uniform(-self.delta, self.delta))
        delta_H = lattice.dhamiltonien(new_theta, idx, epsilon, gamma, A)

        self.attempts += 1
        if delta_H < 0 or np.random.rand() < np.exp(-beta * delta_H):
            lattice.theta[i, j] = new_theta
            self.acceptances += 1

        if tune and self.attempts % self.tune_interval == 0:
            acceptance_rate = self.acceptances / max(1, self.attempts)
            if acceptance_rate < self.target_acceptance - 0.05:
                self.delta = max(self.min_delta, self.delta * self.shrink_factor)
            elif acceptance_rate > self.target_acceptance + 0.05:
                self.delta = min(self.max_delta, self.delta * self.growth_factor)
            self.attempts = 0
            self.acceptances = 0

    def overrelaxation_step(self, lattice, epsilon, gamma, A, beta):
        '''
        Perform one overrelaxation step
        Inputs:
            lattice: Lattice object
            epsilon, gamma, A: parameters
            beta
        '''

        for idx, (i, j) in enumerate(lattice.indices):
            sum_sin6 = 0.0
            sum_cos6 = 0.0
            for (ni, nj) in lattice.voisins[idx]:
                sum_cos6 += np.cos(6 * lattice.theta[ni, nj])
                sum_sin6 += np.sin(6 * lattice.theta[ni, nj])
            
            if sum_cos6 == 0 and sum_sin6 == 0:
                continue

            phi6 = np.arctan2(sum_sin6, sum_cos6)
            theta_old = lattice.theta[i, j]
            theta_new = wrap_angle((2 * phi6 / 6) - theta_old)
            delta_H = lattice.dhamiltonien(theta_new, idx, epsilon, gamma, A)
            if delta_H < 0 or np.random.rand() < np.exp(-beta * delta_H):
                lattice.theta[i, j] = theta_new

    def cluster_step(self, lattice, epsilon, gamma, A, beta, tol=1e-2):
        '''
        Perform one cluster update step 
        Usefull for low temperatures
        Inputs:
            lattice: Lattice object
            epsilon, gamma, A: parameters
            beta
        '''

        L = lattice.L
        N = L * L

        seed_idx = np.random.randint(N)

        in_cluster = np.zeros(N, dtype=bool)
        cluster_idx = []

        queue = [seed_idx]
        in_cluster[seed_idx] = True
        angle_neighbor = []

        while queue:
            current_idx = queue.pop()
            cluster_idx.append(current_idx)
            
            i, j = lattice.indices[current_idx]

            for (ni, nj) in lattice.voisins[current_idx]:
                neighbor_idx = lattice.index_map[ni, nj]
                if not in_cluster[neighbor_idx]:
                    dtheta = np.abs(wrap_angle(lattice.theta[i, j] - lattice.theta[ni, nj]))
                    if dtheta < tol:
                        in_cluster[neighbor_idx] = True
                        queue.append(neighbor_idx)
                    else:
                        angle_neighbor.append(dtheta)
                        

        delta_theta = np.median(angle_neighbor) if angle_neighbor else 0.0

        dH_0 = 0.0
        dH_int = 0.0

        for idx in cluster_idx:
            i, j = lattice.indices[idx]
            old_theta = lattice.theta[i, j]
            new_theta = wrap_angle(old_theta + delta_theta)
            
            dH_0 += (epsilon / lattice.rho) * (new_theta**2 - old_theta**2)

            for (ni, nj) in lattice.voisins[idx]:
                neighbor_idx = lattice.index_map[ni, nj]
                if not in_cluster[neighbor_idx]:
                    dtheta_old = np.abs(wrap_angle(old_theta - lattice.theta[ni, nj]))
                    dtheta_new = np.abs(wrap_angle(new_theta - lattice.theta[ni, nj]))
                    dH_int += dtheta_new * (A - np.log(max(dtheta_new, 1e-12))) - dtheta_old * (A - np.log(max(dtheta_old, 1e-12)))

        dH_int = (gamma / 3 * np.sqrt(lattice.rho)) * dH_int
        delta_H = dH_0 + dH_int

        if delta_H < 0 or np.random.rand() < np.exp(-beta * delta_H):
            for idx in cluster_idx:
                i, j = lattice.indices[idx]
                lattice.theta[i, j] = wrap_angle(lattice.theta[i, j] + delta_theta)
                    
    
class Simulation:
    def __init__(self, L, T, epsilon, gamma, A, rho=1.0, n_thermal=50_000, n_steps=10_000, overrelax_interval=1000, cluster_interval=1000, tol_cluster=1e-2, measure_interval=1000, tune_interval=100_000, tune_delta=True, delta_init=0.2, target_acceptance=0.5, growth_factor=1.05, shrink_factor=0.95, min_delta=1e-3, max_delta=np.pi/6):
        self.L = L
        self.T = T
        self.epsilon = epsilon
        self.gamma = gamma
        self.A = A
        self.beta = 1.0 / T
        self.rho = rho
        self.n_thermal = n_thermal
        self.n_steps = n_steps
        self.overrelax_interval = overrelax_interval
        self.cluster_interval = cluster_interval
        self.tol_cluster = tol_cluster
        self.measure_interval = measure_interval
        self.lattice = Lattice(L, rho)
        self.metropolis = Metropolis(delta_init=delta_init, target_acceptance=target_acceptance, tune_interval=tune_interval, growth_factor=growth_factor, shrink_factor=shrink_factor, min_delta=min_delta, max_delta=max_delta)
        self.tune_delta = tune_delta
        self.energies = []
        self.r = self.lattice.r_centers
        self.G = None
        self.G_err = None
        self.chi6 = None
        self.chi6_err = None

    def save_chi6(self, path):
        parent = Path(path).parent
        outfile = Path(f"{parent}/chi6_L{self.L}.csv")

        if not outfile.exists():
            with open(outfile, 'w') as f:
                f.write("T,chi6,chi6_error\n")
                f.write(f"{self.T:.8e},{self.chi6:.8e},{self.chi6_err:.8e}\n")
            return
            
        data = np.loadtxt(outfile, delimiter=",", skiprows=1)
        if data.ndim == 1:
            data = data.reshape(1, -1)

        T_values = data[:, 0]

        mask = np.isclose(T_values, self.T, atol=1e-8)
        if np.any(mask):
            data[mask, 1] = self.chi6
            data[mask, 2] = self.chi6_err
        else:
            new_row = np.array([[self.T, self.chi6, self.chi6_err]])
            data = np.vstack((data, new_row))

        sorted_indices = np.argsort(data[:, 0])
        data = data[sorted_indices]

        with open(outfile, 'w') as f:
            f.write("T,chi6,chi6_error\n")
            for row in data:
                f.write(f"{row[0]:.8e},{row[1]:.8e},{row[2]:.8e}\n")
    
    def save_results(self, path):
        '''
        Save simulation results to files
        Inputs:
            path: Directory path to save the results
        '''
        
        os.makedirs(path, exist_ok=True)

        theta = theta_flatten(self.lattice.theta, self.lattice.indices)
        x_positions = self.lattice.xs
        y_positions = self.lattice.ys

        header = "x,y,theta"
        data = np.column_stack((x_positions, y_positions, theta))
        np.savetxt(f"{path}/Final_Configuration_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

        header = "step,energy_per_site"
        data = np.column_stack((np.arange(len(self.energies)) * (self.n_thermal // 10), self.energies))
        np.savetxt(f"{path}/Energy_Evolution_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

        header = "r,G,G_error"
        data = np.column_stack((self.r, self.G, self.G_err))
        np.savetxt(f"{path}/Correlation_Function_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

        self.save_chi6(path)

    def compute_susceptibility(self, Psis):
        '''
        Compute the susceptibility of the lattice
        '''

        N = self.L * self.L

        self.chi6 = N * self.beta * (np.mean(np.abs(Psis)**2) - np.abs(np.mean(Psis))**2)
        self.chi6_err = N * self.beta * np.std(np.abs(Psis)**2) / np.sqrt(len(Psis))

    def one_step(self, step, tune=True, thermal=True):
        '''
        Perform one simulation step (Metropolis + Overrelaxation)
        '''
        if thermal:
            if self.overrelax_interval > 0 and step % self.overrelax_interval == 0:
                self.metropolis.overrelaxation_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta)
            if self.cluster_interval > 0 and step % self.cluster_interval == 0:
                self.metropolis.cluster_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta, tol=self.tol_cluster)
            else:
                self.metropolis.metropolis_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta, tune=tune)
        else:
            self.metropolis.metropolis_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta, tune=False)

    def run(self, path):
        '''
        Run the Monte Carlo simulation
        Inputs:
            path: Directory path to save the results
        '''
        
        display(Markdown(f"## Running Simulation at T={self.T:.2e}, L={self.L}"))

        for step in range(self.n_thermal + 1):
            self.one_step(step, tune=self.tune_delta)
            if step % (self.n_thermal // 10) == 0:
                energy = self.lattice.hamiltonien(self.epsilon, self.gamma, self.A) / (self.L * self.L)
                self.energies.append(energy)
        
        Gs = []
        Phis = []

        for step in range(self.n_steps + 1):
            self.one_step(step, thermal=False)
            if step % (self.n_steps // 100) == 0:
                Phi = self.lattice.compute_magnetization()
                Phis.append(Phi)
            if self.measure_interval > 0 and step % self.measure_interval == 0:
                G = self.lattice.compute_orientationnal_correlation()
                Gs.append(G)

        self.G = np.mean(Gs, axis=0)
        self.G_err = np.std(Gs, axis=0) / np.sqrt(len(Gs))
        self.compute_susceptibility(np.array(Phis))

        rate = (self.metropolis.acceptances /
                max(1, self.metropolis.attempts))

        display(Markdown(f"### Final acceptance rate: {rate:.3f}, Final delta: {self.metropolis.delta:.4f} rad"))
        self.save_results(path)

### Plot results

In [ ]:
def plot_results(path, L, T, epsilon, rho):
    '''
    Plot the results of the simulation
    Inputs:
        path: path to the directory where the results are saved
        L: lattice size
        T: temperature
    '''
    
    theta_data = np.loadtxt(f"{path}/Final_Configuration_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    x_positions = theta_data[:, 0]
    y_positions = theta_data[:, 1]
    theta = theta_data[:, 2]

    energy_data = np.loadtxt(f"{path}/Energy_Evolution_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    steps = energy_data[:, 0]
    energies = energy_data[:, 1]

    corr_data = np.loadtxt(f"{path}/Correlation_Function_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    r = corr_data[:, 0]
    G = corr_data[:, 1]
    G_err = corr_data[:, 2]

    mask = np.abs(G) > 0
    r = r[mask]
    G = G[mask]
    G_err = G_err[mask]

    plt.figure(figsize=(8, 6))
    plt.scatter(x_positions, y_positions, c=theta, cmap='hsv', s=40, marker='h', vmin=-np.pi/6, vmax=np.pi/6)
    plt.colorbar(label=r'$\theta$ (rad)')
    plt.title(rf'Hexagonal Lattice L={L}, $T={T:.2e}$')
    plt.xlabel(r'$x$')
    plt.ylabel(r'$y$')
    plt.savefig(f"{path}/Final_Configuration_T{T:.2e}_L{L}.pdf")
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.plot(steps, energies)
    plt.title(rf'Energy per Spin at $T={T:.2e}$, $L={L}$')
    plt.xlabel('Monte Carlo Steps')
    plt.ylabel(r'$E/N$')
    plt.grid()
    plt.savefig(f"{path}/Energy_Evolution_T{T:.2e}_L{L}.pdf")
    plt.close()

    r_min = r[1]
    r_max = r[-1] / 2
    mask = (r >= r_min) & (r <= r_max) & (G > 0)
    coef_power_decay, cov_power_decay = np.polyfit(np.log(r[mask]), np.log(G[mask]), 1, cov=True)
    a_power, b_power = coef_power_decay
    eta6 = -a_power
    eta6_err = np.sqrt(cov_power_decay[0, 0])
    A_power = np.exp(b_power)
    G_power_fit = A_power * r[1:]**(-eta6)
    G_power_fit = np.insert(G_power_fit, 0, G[0])

    coef_expo_decay, cov_expo_decay = np.polyfit(r[mask], np.log(G[mask]), 1, cov=True)
    a_expo, b_expo = coef_expo_decay
    xi6 = -1 / a_expo
    A = np.exp(b_expo)
    xi6_err = xi6**2 * np.sqrt(cov_expo_decay[0, 0])
    G_expo_fit = A * np.exp(-r/xi6)

    # G_th = np.exp(-18 * rho * T / epsilon)

    plt.figure(figsize=(8, 6))
    plt.errorbar(r, G, yerr=G_err, fmt='-x', label='Simulation Data')
    plt.plot(r, G_power_fit, linestyle='--', label=rf'Power-law fit: $\eta_6={eta6:.3f} \pm {eta6_err:.3f}$')
    plt.plot(r, G_expo_fit, linestyle=':', label=rf'Exponential fit: $\xi_6={xi6:.2f} \pm {xi6_err:.2f}$')
    # plt.axhline(G_th, color='black', linestyle='-.', label=rf'Theoretical $G_6(r)$')
    plt.title(rf'Correlation Function at $T={T:.2e}$, $L={L}$')
    plt.xlabel(r'$r$')
    plt.ylabel(r'$G_6(r)$')
    plt.ylim(-0.2,1.02)
    plt.legend()
    plt.grid()
    plt.savefig(f"{path}/Correlation_Function_T{T:.2e}_L{L}.pdf")
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.errorbar(r, G, yerr=G_err, fmt='-x', label='Simulation Data')
    plt.plot(r, G_power_fit, linestyle='--', label=rf'Power-law fit: $\eta_6={eta6:.3f} \pm {eta6_err:.3f}$')
    plt.plot(r, G_expo_fit, linestyle=':', label=rf'Exponential fit: $\xi_6={xi6:.2f} \pm {xi6_err:.2f}$')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(rf'Correlation Function at $T={T:.2e}$, $L={L}$ (Log-Log Scale)')
    plt.xlabel(r'$r$')
    plt.ylabel(r'$G_6(r)$')
    plt.legend()
    plt.grid()
    plt.savefig(f"{path}/Correlation_Function_T{T:.2e}_L{L}_loglog.pdf")
    plt.close()

def plot_chi6_vs_T(path, L):
    '''
    Plot chi6 vs T from multiple simulations
    Inputs:
        path: path to the directory where the results are saved
        L: lattice size
    '''

    chi6_data = np.loadtxt(f"{path}/chi6_L{L}.csv", delimiter=",", skiprows=1)
    T_values = chi6_data[:, 0]
    chi6 = chi6_data[:, 1]
    chi6_err = chi6_data[:, 2]
    
    plt.figure(figsize=(8, 6))
    plt.errorbar(T_values, chi6, yerr=chi6_err, fmt='-x', label='Simulation Data')
    plt.title(rf'Susceptibility $\chi_6$ vs Temperature for $L={L}$')
    plt.xlabel(r'$T$')
    plt.ylabel(r'$\chi_6$')
    plt.grid()
    plt.savefig(f"{path}/chi6_L{L}.pdf")
    plt.close()

## Simulation 

In [5]:
epsilon = 0
gamma = 1.0
rho = 1.0
A = 1.0

n_thermal=5*10**6
n_steps=10**4
measure_interval=n_steps // 10
tune_interval=n_thermal // 100
overrelax_interval=n_thermal // 200
cluster_interval=0
tol_cluster=0.01

T = [0.19]
L=32
for temp in T:
    path = f"Epsilon{epsilon}Gamma{gamma}/L{L}/T{temp:.2e}"
    sim = Simulation(L=L, T=temp, epsilon=epsilon, gamma=gamma, A=A, rho=rho, n_thermal=n_thermal, n_steps=n_steps, overrelax_interval=overrelax_interval, cluster_interval=cluster_interval, tol_cluster=tol_cluster, measure_interval=measure_interval, tune_interval=tune_interval, tune_delta=True)
    sim.run(path=path)

winsound.Beep(500, 2000)

## Running Simulation at T=1.90e-01, L=32

KeyboardInterrupt: 

In [6]:
epsilon = 0
gamma = 1.0
rho = 1.0
T = [10**(-5), 0.01, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.21, 0.22]
L=32
for temp in T:
    path = f"Epsilon{epsilon}Gamma{gamma}/L{L}/T{temp:.2e}"
    plot_results(path, L, temp, epsilon, rho)

path = f"Epsilon{epsilon}Gamma{gamma}/L{L}"
plot_chi6_vs_T(path, L)